In [1]:
import camp_tv_new.src.utils.database_utils as db_utils
import pandas as pd
import camp_tv_new.src.utils.bereik_utils as bereik_utils
import camp_tv_new.src.utils.database_utils as db_utils

import pandas as pd


In [2]:
df_flights = db_utils.query_to_df('SELECT flight_id, advertiser, product, start_date, end_date, campaign_duration, mock_campaign FROM [dbo].[tv.flights]')
df_flights = df_flights[df_flights['mock_campaign'] == 0]
unique_flight_ids = set(df_flights['flight_id'])

In [3]:
mylist = ['25-67', 'B2B', '20-49', '3-12', '30-65', '35+', '35-59', '35-64', '50+', '6+', '68+',
            '7-12', 'BDS 25-59', 'BDS 25-67', '13+', '25+', '25-39', '25-54', '25-59', '25-59AB1', '25-67AB1',
            'BDS KZ12', 'BDS+KND', 'DAG 25-59', 'M 20-49', 'M 25-54', 'M 25-59',
            'M 25-67', 'M 50+', 'V 20-49', 'V 25-54', 'V 25-59', 'V 25-67', 'V 40+',
            'V 50+', 'W1 45-75']

rename_dict = {
    'M 20-49': 'M20-49',
    'M 25-54': 'M25-54',
    'M 25-59': 'M25-59',
    'M 25-67': 'M25-67',
    'M 50+': 'M50+',
    'V 20-49': 'V20-49',
    'V 25-54': 'V25-54',
    'V 25-59': 'V25-59',
    'V 25-67': 'V25-67',
    'V 40+': 'V40+',
    'V 50+': 'V50+',
    'W1 45-75': 'W145-75'
}

for target_group in mylist:
    print(target_group)
    target_group_db = rename_dict.get(target_group, target_group)
    query = f"""SELECT DISTINCT flight_id, target_group, campaign_duration FROM [dbo].[tv.bereik] where target_group = '{target_group_db}'"""
    bereik_data_tg = db_utils.query_to_df(query)
    df_db_flight_ids = set(bereik_data_tg['flight_id'])

    # Identify new flights
    missing_flights_in_db = list(unique_flight_ids - df_db_flight_ids)
    print('missing flights:', len(missing_flights_in_db))

    df = pd.read_excel(f"../../data/raw/api/bereik/{target_group}/{target_group}_bereik_all_flights.xlsx", dtype={'start_date': str, 'end_date': str})

    processed_rows = [bereik_utils.process_row(row) for _, row in df.iterrows()]
    processed_df = pd.DataFrame(processed_rows).fillna(0)
    # Calculate campaign duration
    processed_df['campaign_duration'] = processed_df.apply(lambda row: bereik_utils.calculate_campaign_duration(row), axis=1)
    # Change column order
    processed_df.insert(1, 'target_group', target_group_db)
    processed_df = bereik_utils.change_column_order(processed_df)

    columns_to_exclude = ['start_date', 'end_date', 'advertiser', 'product']
    columns_to_include = [col for col in processed_df.columns if col not in columns_to_exclude]
    processed_df = processed_df[columns_to_include]

    df_missing = processed_df[processed_df['flight_id'].isin(missing_flights_in_db)]
    print('df_missing:', len(df_missing))

    batch_size = 10
    for i in range(0, len(df_missing), batch_size):
        batch = df_missing[i:i + batch_size]
        db_utils.dump_to_sql(batch, table_name = 'tv.bereik')


25-67
missing flights: 83
df_missing: 49
B2B
missing flights: 3250
df_missing: 855
20-49
missing flights: 3252
df_missing: 861
3-12
missing flights: 3290
df_missing: 973
30-65
missing flights: 3290
df_missing: 973
35+
missing flights: 3290
df_missing: 973
35-59
missing flights: 3290
df_missing: 973
35-64
missing flights: 3290
df_missing: 973
50+
missing flights: 3290
df_missing: 973
6+
missing flights: 5630


FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/api/bereik/6+/6+_bereik_all_flights.xlsx'

In [21]:
import camp_tv_new.src.utils.connection_utils as con_utils
from sqlalchemy import delete, text

def delete_flights(list_of_flights, table_name):
    try:
        # Connect to the database
        engine = con_utils.create_sql_alchemy_engine()
        # Convert list_of_flights to a tuple for use in the SQL query
        list_of_flights = tuple(list_of_flights)

        # Construct the SQL DELETE statement
        sql_delete = f"""
        DELETE FROM [dbo].{table_name}
        WHERE flight_id IN {list_of_flights}
        """
        with engine.connect() as connection:
            # Begin a transaction
            with connection.begin() as transaction:
                try:
                    # Execute the DELETE statement
                    connection.execute(text(sql_delete))
                    # Commit the transaction
                    transaction.commit()
                except Exception as e:
                    # Rollback the transaction in case of error
                    transaction.rollback()

                    raise

    except Exception as e:
        print(f"Error deleting flights: {e}")

In [22]:
list_of_flights = ['e5016d48-ab0f-4434-de6b-5619f552805c', 'e5016d48-ab0f-4434-de6b-5619f552805c']
delete_flights(list_of_flights, '[tv.bereik]')